# Scientific Uncertainty Detection in Biomedical Text Using Machine Learning and Transformer-Based NLP Models.

## Preprocessing
### Step 1: Case Folding

In [1]:
import pandas as pd

train_df = pd.read_csv("Dataset/train.csv")
dev_df = pd.read_csv("Dataset/dev.csv")
test_df = pd.read_csv("Dataset/test.csv")


def case_fold(text):
    tokens = text.split()
    normalized_tokens = [x.lower() for x in tokens]
    return " ".join(normalized_tokens)


for df in (train_df, dev_df, test_df):
    df["text_lower"] = df["text"].apply(case_fold)

print(train_df.shape, dev_df.shape, test_df.shape)
# print(train_df[["text", "text_lower"]].to_string())

(10175, 7) (1291, 7) (1491, 7)


### Step 2: Synonym Substitution (Not Used)
Synonym substitution can replace cue words such as `may`, `suggest` and `possible`, which changes the uncertainty label of a sentence. It is therefore skipped.

### Step 3: Stemming (Not Used)
Stemming produces non-words (`possible` → `possibl`) and damages biomedical terms. Lemmatization is used instead.

### Step 4: Lemmatization

In [2]:
import spacy

nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])


def spacy_lemmatize(texts):
    return [" ".join(tok.lemma_ for tok in doc) for doc in nlp.pipe(texts, batch_size=256)]


for df in (train_df, dev_df, test_df):
    df["text_lemma"] = spacy_lemmatize(df["text_lower"].tolist())

### Step 5: Punctuation Removal

In [3]:
import string

punct_mapping = str.maketrans("", "", string.punctuation)


def remove_punctuation(text):
    return text.lower().translate(punct_mapping).strip()


for df in (train_df, dev_df, test_df):
    df["text_clean"] = df["text_lemma"].apply(remove_punctuation)